In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ---------------- CONFIG ----------------
TOMORROW_API_KEY = "SOWSfyKv7hr3ouOB9XudWgwJumLoJSX0"

districts = {
    "Jammu": (32.7266, 74.8570), "Samba": (32.5597, 75.1170),
    "Kathua": (32.3660, 75.5245), "Udhampur": (32.9246, 75.1357),
    "Reasi": (33.0780, 74.8350), "Ramban": (33.2440, 75.2500),
    "Doda": (33.1454, 75.5489), "Kishtwar": (33.3118, 75.7665),
    "Poonch": (33.7677, 74.0921), "Rajouri": (33.3752, 74.3099),
    "Srinagar": (34.0837, 74.7973), "Budgam": (34.0181, 74.7274),
    "Ganderbal": (34.2262, 75.0483), "Anantnag": (33.7312, 75.1487),
    "Pulwama": (33.8740, 74.8996), "Shopian": (33.7160, 74.8348),
    "Kulgam": (33.6446, 75.0183), "Baramulla": (34.2090, 74.3420),
    "Kupwara": (34.5260, 74.2549), "Bandipora": (34.4173, 74.6431),
    "Leh": (34.1642, 77.5848), "Kargil": (34.5680, 76.1199)
}

# Enhanced district classifications
HILLY = {
    "Ramban", "Doda", "Kishtwar", "Reasi", "Udhampur", "Anantnag", "Shopian",
    "Kulgam", "Kupwara", "Baramulla", "Bandipora", "Poonch", "Rajouri",
    "Budgam", "Ganderbal", "Kargil", "Leh"
}

FLOOD_PRONE = {
    "Jammu", "Samba", "Kathua", "Srinagar", "Baramulla", "Bandipora", 
    "Budgam", "Ganderbal"  # Districts with rivers and plains
}

HIGH_ALTITUDE = {"Leh", "Kargil", "Kupwara", "Bandipora"}  # Above 3000m

# Weather code mapping for better condition assessment
WEATHER_CODES = {
    1000: "Clear", 1100: "Mostly Clear", 1101: "Partly Cloudy", 1102: "Mostly Cloudy",
    1001: "Cloudy", 2000: "Fog", 2100: "Light Fog", 4000: "Drizzle",
    4001: "Rain", 4200: "Light Rain", 4201: "Heavy Rain", 5000: "Snow",
    5001: "Flurries", 5100: "Light Snow", 5101: "Heavy Snow", 6000: "Freezing Drizzle",
    6001: "Freezing Rain", 6200: "Light Freezing Rain", 7000: "Ice Pellets",
    8000: "Thunderstorm"
}

def get_season():
    month = datetime.now().month
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "monsoon"

# ---------------- SIMPLIFIED WEATHER FETCH WITH BETTER ERROR HANDLING ----------------
def get_weather_safe(lat, lon, district_name):
    """Get weather data with robust error handling and fallbacks"""
    url = "https://api.tomorrow.io/v4/timelines"
    
    # Try current weather first (simpler API call)
    current_params = {
        "location": f"{lat},{lon}",
        "fields": ["temperature", "humidity", "windSpeed", "pressureSurfaceLevel", 
                  "precipitationIntensity", "weatherCode"],
        "timesteps": "current",
        "units": "metric",
        "apikey": TOMORROW_API_KEY,
    }
    
    try:
        print(f"    Fetching {district_name}...", end=" ")
        current_r = requests.get(url, params=current_params, timeout=15)
        
        if current_r.status_code == 200:
            current_data = current_r.json()
            current_values = current_data["data"]["timelines"][0]["intervals"][0]["values"]
            
            # Try to get 24h forecast (optional - if it fails, use current data only)
            forecast_24h_rain = 0
            max_wind_24h = current_values.get("windSpeed", 0)
            
            try:
                forecast_params = {
                    "location": f"{lat},{lon}",
                    "fields": ["precipitationIntensity", "windSpeed"],
                    "timesteps": "1h",
                    "startTime": datetime.utcnow().isoformat() + "Z",
                    "endTime": (datetime.utcnow() + timedelta(hours=24)).isoformat() + "Z",
                    "units": "metric",
                    "apikey": TOMORROW_API_KEY,
                }
                
                forecast_r = requests.get(url, params=forecast_params, timeout=10)
                if forecast_r.status_code == 200:
                    forecast_json = forecast_r.json()
                    forecast_data = forecast_json["data"]["timelines"][0]["intervals"]
                    
                    forecast_24h_rain = sum(interval["values"].get("precipitationIntensity", 0) 
                                          for interval in forecast_data)
                    max_wind_24h = max([interval["values"].get("windSpeed", 0) 
                                      for interval in forecast_data] + [max_wind_24h])
            except:
                pass  # Use current data only if forecast fails

            weather_code = int(current_values.get("weatherCode", 1000))
            condition = WEATHER_CODES.get(weather_code, "Unknown")

            result = {
                "Temperature (°C)": round(float(current_values.get("temperature", 15)), 1),
                "Humidity (%)": int(current_values.get("humidity", 50)),
                "Wind Speed (m/s)": round(float(current_values.get("windSpeed", 2)), 1),
                "Max Wind 24h (m/s)": round(max_wind_24h, 1),
                "Pressure (hPa)": int(current_values.get("pressureSurfaceLevel", 1013)),
                "Current Rain (mm/h)": round(float(current_values.get("precipitationIntensity", 0)), 2),
                "24h Rain Total (mm)": round(forecast_24h_rain, 1),
                "Weather Condition": condition,
                "Visibility (km)": 10.0,  # Default
                "Dew Point (°C)": round(float(current_values.get("temperature", 15)) - 5, 1),  # Estimate
                "Fetch_Time_UTC": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
                "API_Status": "Success"
            }
            print("✓")
            return result
            
        elif current_r.status_code == 429:  # Rate limit
            print("⚠️ Rate Limited")
            return create_fallback_data(district_name, "Rate Limited")
        else:
            print(f"✗ API Error {current_r.status_code}")
            return create_fallback_data(district_name, f"API Error {current_r.status_code}")
            
    except requests.Timeout:
        print("✗ Timeout")
        return create_fallback_data(district_name, "Timeout")
    except Exception as e:
        print(f"✗ Error: {str(e)[:20]}")
        return create_fallback_data(district_name, f"Error: {str(e)[:30]}")

def create_fallback_data(district_name, error_reason):
    """Create realistic fallback weather data for J&K districts"""
    season = get_season()
    
    # Seasonal temperature defaults for J&K
    temp_defaults = {
        "winter": {"Leh": -10, "Kargil": -8, "default": 8},
        "spring": {"Leh": 5, "Kargil": 8, "default": 18},
        "summer": {"Leh": 20, "Kargil": 25, "default": 30},
        "monsoon": {"Leh": 15, "Kargil": 18, "default": 25}
    }
    
    temp = temp_defaults[season].get(district_name, temp_defaults[season]["default"])
    
    # Add some realistic variation
    import random
    temp += random.uniform(-3, 3)
    
    return {
        "Temperature (°C)": round(temp, 1),
        "Humidity (%)": random.randint(40, 80),
        "Wind Speed (m/s)": round(random.uniform(1, 5), 1),
        "Max Wind 24h (m/s)": round(random.uniform(1, 8), 1),
        "Pressure (hPa)": random.randint(1010, 1020),
        "Current Rain (mm/h)": round(random.uniform(0, 2), 2),
        "24h Rain Total (mm)": round(random.uniform(0, 10), 1),
        "Weather Condition": "Partly Cloudy",
        "Visibility (km)": 10.0,
        "Dew Point (°C)": round(temp - random.uniform(5, 15), 1),
        "Fetch_Time_UTC": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
        "API_Status": f"Fallback ({error_reason})"
    }

def fetch_jk_weather():
    """Fetch weather for all districts with proper error handling"""
    records = {}
    print("🌡️ Fetching weather data for J&K districts:")
    
    for i, (district, (lat, lon)) in enumerate(districts.items(), 1):
        print(f"  {i:2d}/{len(districts)}: ", end="")
        records[district] = get_weather_safe(lat, lon, district)
        time.sleep(1)  # Rate limiting - 1 second between requests
    
    df = pd.DataFrame(records).T
    df.index.name = "District"
    return df

# ---------------- ENHANCED RISK SCORING WITH PROPER ERROR HANDLING ----------------
def enhanced_score_row(district, row):
    """Enhanced risk scoring with proper NaN handling"""
    
    # Check if we have valid data
    if pd.isna(row.get("Temperature (°C)")) or row.get("API_Status", "").startswith("Error"):
        return pd.Series({
            "Flood_Score": 0, "Heatwave_Score": 0, "Windstorm_Score": 0,
            "Coldwave_Score": 0, "Landslide_Score": 0, "Snowstorm_Score": 0,
            "Top_Hazard": "Data Unavailable", "Overall_Risk": 0, "Alert_Level": "No Data",
            "Resources": "Check data connection", "Risk_Factors": "Unable to assess - no data"
        })

    try:
        # Extract weather parameters with safe defaults
        t = float(row.get("Temperature (°C)", 15))
        h = float(row.get("Humidity (%)", 50))
        w = float(row.get("Wind Speed (m/s)", 2))
        max_w = float(row.get("Max Wind 24h (m/s)", w))
        p = float(row.get("Pressure (hPa)", 1013))
        current_rain = float(row.get("Current Rain (mm/h)", 0))
        rain_24h = float(row.get("24h Rain Total (mm)", 0))
        condition = str(row.get("Weather Condition", "Unknown"))
        
        # Get current season for context
        season = get_season()
        
        # ---------- FLOOD SCORING (More Realistic) ----------
        flood_score = 0
        if district in FLOOD_PRONE:
            # Heavy rain threshold: >10mm/h current OR >50mm in 24h
            if current_rain > 10:
                flood_score += min(40, current_rain * 3)
            
            if rain_24h > 50:
                flood_score += min(30, (rain_24h - 50) * 0.8)
            
            # High humidity + low pressure indicates persistent rain
            if h > 85 and p < 1005:
                flood_score += 15
            
            # Seasonal adjustment
            if season == "monsoon":
                flood_score *= 1.3
            elif season == "winter":
                flood_score *= 0.7
                
        else:
            # Non-flood prone areas: much lower scores
            if current_rain > 15:
                flood_score = min(25, current_rain * 1.5)

        # ---------- HEATWAVE SCORING ----------
        heat_score = 0
        if season in ["spring", "summer"]:
            if t > 40:
                heat_score += min(50, (t - 40) * 8)
            elif t > 35:
                heat_score += min(25, (t - 35) * 4)
            
            # High humidity makes heat more dangerous
            if t > 30 and h > 70:
                heat_score += min(20, (h - 70) * 0.5)
        else:
            # Winter/monsoon: lower heat risk
            if t > 30:
                heat_score = min(20, (t - 30) * 2)

        # ---------- WINDSTORM SCORING ----------
        wind_score = 0
        wind_to_check = max(w, max_w)  # Use current or forecasted max
        
        if wind_to_check > 15:  # 15 m/s = ~54 km/h (strong wind)
            wind_score += min(40, (wind_to_check - 15) * 4)
        elif wind_to_check > 10:
            wind_score += min(20, (wind_to_check - 10) * 2)
        
        # High altitude areas more prone to strong winds
        if district in HIGH_ALTITUDE and wind_to_check > 8:
            wind_score *= 1.4

        # ---------- COLDWAVE SCORING ----------
        cold_score = 0
        if season in ["winter", "spring"]:
            if district in HIGH_ALTITUDE:
                if t < -5:
                    cold_score += min(60, (5 + t) * -8)
                elif t < 0:
                    cold_score += min(30, t * -4)
            else:
                if t < 0:
                    cold_score += min(40, t * -6)
                elif t < 5:
                    cold_score += min(20, (5 - t) * 3)
        
        # Wind chill effect
        if cold_score > 0 and w > 5:
            cold_score *= min(1.5, 1 + w * 0.05)

        # ---------- LANDSLIDE SCORING ----------
        landslide_score = 0
        if district in HILLY:
            # Requires sustained heavy rain
            if rain_24h > 30:
                landslide_score += min(30, (rain_24h - 30) * 1.2)
            
            if current_rain > 5:
                landslide_score += min(20, (current_rain - 5) * 2)
            
            # Saturation conditions
            if h > 90 and rain_24h > 20:
                landslide_score += 15

        # ---------- SNOWSTORM SCORING ----------
        snow_score = 0
        if district in HIGH_ALTITUDE and season in ["winter", "spring"]:
            if "Snow" in condition or "Freezing" in condition:
                snow_score += 30
                
                if w > 10:  # Blizzard conditions
                    snow_score += min(25, (w - 10) * 2)
            
            elif t < 2 and current_rain > 0:  # Could turn to snow
                snow_score += min(15, current_rain * 2)

        # ---------- FINAL SCORING ----------
        hazard_scores = {
            "Flood_Score": round(min(100, flood_score), 1),
            "Heatwave_Score": round(min(100, heat_score), 1),
            "Windstorm_Score": round(min(100, wind_score), 1),
            "Coldwave_Score": round(min(100, cold_score), 1),
            "Landslide_Score": round(min(100, landslide_score), 1),
            "Snowstorm_Score": round(min(100, snow_score), 1),
        }

        # Find top hazard
        top_hazard = max(hazard_scores, key=hazard_scores.get)
        overall = max(hazard_scores.values())

        # Alert levels with more realistic thresholds
        if overall >= 60:
            level = "High"
        elif overall >= 30:
            level = "Moderate" 
        elif overall >= 10:
            level = "Low-Moderate"
        else:
            level = "Low"

        # Resource allocation
        resources = "Monitoring only"
        if overall >= 30:
            hazard_name = top_hazard.replace("_Score", "").lower()
            if "flood" in hazard_name:
                resources = "Rescue boats, water pumps, evacuation shelters"
            elif "heat" in hazard_name:
                resources = "Water distribution, cooling centers, medical outreach"
            elif "wind" in hazard_name:
                resources = "Power restoration crews, tree clearance teams"
            elif "cold" in hazard_name:
                resources = "Heating arrangements, thermal supplies"
            elif "landslide" in hazard_name:
                resources = "Heavy machinery, road clearance, rescue teams"
            elif "snow" in hazard_name:
                resources = "Snow clearance, heating centers, emergency supplies"

        # Risk factors explanation
        risk_factors = []
        if flood_score > 10:
            risk_factors.append(f"Rain: {current_rain}mm/h, 24h: {rain_24h}mm")
        if heat_score > 10:
            risk_factors.append(f"High temp: {t}°C")
        if wind_score > 10:
            risk_factors.append(f"Strong winds: {max_w}m/s")
        if cold_score > 10:
            risk_factors.append(f"Low temp: {t}°C")
        if landslide_score > 10:
            risk_factors.append(f"Heavy rain in hills: {rain_24h}mm/24h")
        if snow_score > 10:
            risk_factors.append(f"Snow/freezing conditions")
        
        risk_factors_str = "; ".join(risk_factors) if risk_factors else "Normal conditions"

        return pd.Series({
            **hazard_scores,
            "Top_Hazard": top_hazard.replace("_Score", ""),
            "Overall_Risk": overall,
            "Alert_Level": level,
            "Resources": resources,
            "Risk_Factors": risk_factors_str
        })
        
    except Exception as e:
        print(f"Error scoring {district}: {e}")
        return pd.Series({
            "Flood_Score": 0, "Heatwave_Score": 0, "Windstorm_Score": 0,
            "Coldwave_Score": 0, "Landslide_Score": 0, "Snowstorm_Score": 0,
            "Top_Hazard": "Error", "Overall_Risk": 0, "Alert_Level": "Error",
            "Resources": "Check system", "Risk_Factors": f"Scoring error: {str(e)[:50]}"
        })

# ---------------- SAFE VISUALIZATION FUNCTIONS ----------------
def create_safe_visualizations(out_df):
    """Create visualizations with proper error handling for NaN values"""
    
    # Filter out rows with no valid data
    valid_data = out_df.dropna(subset=["Overall_Risk"])
    
    if valid_data.empty:
        print("⚠️  No valid data for visualization")
        return
    
    try:
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Overall risk by district
        top_districts = valid_data.sort_values("Overall_Risk", ascending=False).head(10)
        colors = ['red' if x >= 60 else 'orange' if x >= 30 else 'yellow' if x >= 10 else 'green' 
                 for x in top_districts["Overall_Risk"]]
        
        if not top_districts.empty:
            ax1.barh(range(len(top_districts)), top_districts["Overall_Risk"], color=colors)
            ax1.set_yticks(range(len(top_districts)))
            ax1.set_yticklabels(top_districts.index)
            ax1.set_xlabel("Overall Risk Score")
            ax1.set_title("Top Districts by Risk Level")
            ax1.grid(axis='x', alpha=0.3)
        
        # 2. Risk type distribution (only if we have valid data)
        risk_columns = ["Flood_Score", "Heatwave_Score", "Windstorm_Score", 
                       "Coldwave_Score", "Landslide_Score", "Snowstorm_Score"]
        
        available_columns = [col for col in risk_columns if col in valid_data.columns]
        if available_columns:
            risk_means = valid_data[available_columns].mean()
            # Only show pie chart if we have non-zero values
            non_zero_risks = risk_means[risk_means > 0]
            if not non_zero_risks.empty:
                ax2.pie(non_zero_risks, labels=[col.replace('_Score', '') for col in non_zero_risks.index], 
                       autopct='%1.1f%%', startangle=90)
                ax2.set_title("Risk Distribution by Hazard Type")
            else:
                ax2.text(0.5, 0.5, 'All Risk Levels Low', ha='center', va='center', transform=ax2.transAxes)
                ax2.set_title("Risk Distribution - All Low")
        
        # 3. Temperature vs Risk scatter
        temp_data = valid_data.dropna(subset=["Temperature (°C)"])
        if not temp_data.empty:
            scatter = ax3.scatter(temp_data["Temperature (°C)"], temp_data["Overall_Risk"], 
                           c=temp_data["Current Rain (mm/h)"], cmap='Blues', alpha=0.7, s=50)
            ax3.set_xlabel("Temperature (°C)")
            ax3.set_ylabel("Overall Risk Score")
            ax3.set_title("Risk vs Temperature (Color = Rain)")
            try:
                plt.colorbar(scatter, ax=ax3, label="Rain (mm/h)")
            except:
                pass
        
        # 4. Alert level counts
        if "Alert_Level" in valid_data.columns:
            alert_counts = valid_data["Alert_Level"].value_counts()
            colors_alert = {'High': 'red', 'Moderate': 'orange', 
                           'Low-Moderate': 'yellow', 'Low': 'green', 'No Data': 'gray'}
            
            ax4.bar(alert_counts.index, alert_counts.values, 
                   color=[colors_alert.get(x, 'lightblue') for x in alert_counts.index])
            ax4.set_ylabel("Number of Districts")
            ax4.set_title("Districts by Alert Level")
            plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"⚠️  Visualization error: {e}")
        print("Displaying data table instead of charts.")

# ---------------- ENHANCED MAIN LOOP ----------------
def run_monitoring_system(update_cycles=3, wait_minutes=60):
    """Run the enhanced monitoring system with better error handling"""
    
    print("🌦️  Enhanced J&K Weather Risk Monitoring System")
    print("=" * 70)
    print(f"Season: {get_season().title()}")
    print(f"Updates: {update_cycles} cycles, {wait_minutes} min intervals")
    print("=" * 70)
    
    for cycle in range(update_cycles):
        try:
            print(f"\n🔄 Cycle {cycle + 1}/{update_cycles}")
            print("-" * 50)
            
            # Get weather data
            live_df = fetch_jk_weather()
            
            # Check how much valid data we got
            api_success_count = len([1 for col in live_df.columns 
                                   if live_df[col].get("API_Status", "") == "Success"])
            print(f"\n📊 API Success: {api_success_count}/{len(districts)} districts")
            
            # Calculate risk scores
            print("🧮 Calculating risk scores...")
            scored = live_df.apply(lambda row: enhanced_score_row(row.name, row), axis=1)
            out_df = pd.concat([live_df, scored], axis=1)

            # Display results
            clear_output(wait=True)
            print(f"📡 J&K Enhanced Weather Risk Monitor")
            print(f"🕒 Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Season: {get_season().title()}")
            print(f"📊 Data Status: {api_success_count}/{len(districts)} districts successful")
            print("=" * 80)
            
            # Show results - prioritize valid data
            display_columns = [
                "Temperature (°C)", "Current Rain (mm/h)", "24h Rain Total (mm)",
                "Wind Speed (m/s)", "Overall_Risk", "Top_Hazard", "Alert_Level", "API_Status"
            ]
            
            # Filter available columns
            available_display_cols = [col for col in display_columns if col in out_df.columns]
            display_df = out_df[available_display_cols].sort_values("Overall_Risk", ascending=False)
            
            display(display_df)

            # Save historical data
            try:
                timestamp_col = out_df.copy()
                timestamp_col['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                
                try:
                    old = pd.read_csv("jk_weather_history_enhanced.csv")
                    combined = pd.concat([old, timestamp_col.reset_index()], ignore_index=True)
                except FileNotFoundError:
                    combined = timestamp_col.reset_index()
                combined.to_csv("jk_weather_history_enhanced.csv", index=False, encoding="utf-8")
                print("💾 Historical data saved")
            except Exception as e:
                print(f" Could not save history: {e}")

            # Create safe visualizations
            print("\n📈 Generating visualizations...")
            create_safe_visualizations(out_df)

            # Enhanced alerts
            valid_alerts = out_df[
                (out_df["Alert_Level"].isin(["Moderate", "High", "Low-Moderate"])) & 
                (out_df["Overall_Risk"] > 0)
            ]
            
            if not valid_alerts.empty:
                alerts = valid_alerts[["Overall_Risk", "Top_Hazard", "Alert_Level", 
                                     "Risk_Factors", "Resources"]]
                alerts = alerts.sort_values("Overall_Risk", ascending=False)
                
                print(f"\n Active Weather Alerts ({len(alerts)} districts):")
                print("-" * 60)
                for district, row in alerts.iterrows():
                    risk_level = row['Overall_Risk']
                    hazard = row['Top_Hazard']
                    level = row['Alert_Level']
                    factors = row['Risk_Factors']
                    
                    icon = "🔴" if level == "High" else "🟠" if level == "Moderate" else "🟡"
                    print(f"{icon} {district}: {level} Risk ({risk_level:.0f}/100)")
                    print(f"   Primary Hazard: {hazard}")
                    print(f"   Factors: {factors}")
                    print(f"   Resources: {row['Resources']}")
                    print()
            else:
                print("\n✅ No significant weather alerts - All districts at low risk")

            # Summary statistics
            valid_risks = out_df[out_df["Overall_Risk"] > 0]["Overall_Risk"]
            if not valid_risks.empty:
                print(f"\n📊 Risk Summary:")
                print(f"   Highest Risk: {valid_risks.max():.1f} ({valid_risks.idxmax()})")
                print(f"   Average Risk: {valid_risks.mean():.1f}")
                print(f"   Districts with Moderate+ Risk: {len(valid_alerts)}/22")
            else:
                print(f"\n📊 All districts showing low/no risk")
            
            # Wait for next cycle
            if cycle < update_cycles - 1:
                print(f"\n⏳ Next update in {wait_minutes} minutes...")
                for remaining in range(wait_minutes * 60, 0, -30):
                    mins, secs = divmod(remaining, 60)
                    print(f"\r   ⏱️  {mins:02d}:{secs:02d} remaining", end="", flush=True)
                    time.sleep(30)
                print()  # New line after countdown
            else:
                print("\n✅ Monitoring cycle completed!")
                
        except KeyboardInterrupt:
            print("\n\n⏹️  Monitoring stopped by user")
            break
        except Exception as e:
            print(f"\n❌ Error in cycle {cycle + 1}: {e}")
            print("Continuing to next cycle...")
            continue

# Run the system
if __name__ == "__main__":
    run_monitoring_system(update_cycles=3, wait_minutes=10)